# Football Era Lab: the same engine as the browser

This notebook sends editable team settings to Node and imports `app/engine/index.js`. Python only reads data, starts that process and displays results. Run all cells from the repository root. The optional automatic draft is a labelled notebook demonstration; the browser lets you make every pick and placement.

Use the catalogue below to choose a manager, formation and fifteen person IDs, or supply fifteen exact club-decade source records. Source labels and model reports remain visible alongside the simulation.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess

import pandas as pd
from IPython.display import display

# R: repository root; G: bundled game data; NODE: executable running the shared engine.
R = Path.cwd()
if not (R / 'data' / 'game.json').exists() and (R.parent / 'data' / 'game.json').exists():
    R = R.parent
if not (R / 'data' / 'game.json').exists():
    raise FileNotFoundError('Run this notebook in the Football Era Lab repository with data/game.json.')
NODE = shutil.which('node')  # Or set an absolute node.exe path here.
if not NODE:
    raise FileNotFoundError('Install Node 20 or later, or set NODE to its executable path.')
G = json.loads((R / 'data' / 'game.json').read_text(encoding='utf-8'))
print('Node:', subprocess.check_output([NODE, '--version'], text=True).strip())
print('Data SHA-256:', hashlib.sha256((R / 'data' / 'game.json').read_bytes()).hexdigest())
print('Data build:', G.get('meta', {}).get('v'))


## Inspect the available data

These counts are read from this bundle. A club-decade card is a person's record in one club and decade; many cards can refer to the same person. The squad lists contain all qualifying supplied records, and can be incomplete historical rosters. Apps and goals are apportioned across a whole stint.

In [ ]:
catalogue = pd.DataFrame([
    {'manager': m['nm'], 'attack_grade': m['ga'], 'defence_grade': m['gd'], 'formations': ', '.join(m['f'])}
    for m in G['managers']
])
display(catalogue)
print('Formations:', ', '.join(G['formations']))
coverage = pd.DataFrame([
    {'decade': D, 'club_decades': sum(c['D'] == D for c in G['combos']),
     'cards': sum(len(v) for k, v in G['cards'].items() if int(k.split(':')[1]) == D),
     'listed_opponents': len(G['opp'].get(str(D), []))}
    for D in G['meta']['DS']
])
display(coverage)
print('Unique people:', len(G['people']))
reports = {}
for name in ('validation', 'calibration'):
    p = R / 'data' / f'{name}.json'
    print(f'\n{name}:')
    if not p.exists():
        print('Report not supplied in this checkout.')
        continue
    reports[name] = json.loads(p.read_text(encoding='utf-8'))
    summary = {k: v for k, v in reports[name].items()
               if k not in ('unmappedEditionClubNames', 'coverage', 'unresolved')}
    print(json.dumps(summary, indent=2, ensure_ascii=False))
    detail = reports[name].get('coverage')
    if isinstance(detail, list) and detail and isinstance(detail[0], dict):
        display(pd.DataFrame(detail))
    omitted = reports[name].get('unmappedEditionClubNames', [])
    if omitted:
        print('Unmapped edition club names:', len(omitted))
    print(f'Full JSON is available in reports[{name!r}].')


## Configure the experiment

Leave both input lists empty for a seeded notebook draft. `person_ids` must contain fifteen distinct Wikidata person IDs. With IDs alone, the bridge prefers a card from the simulation decade, then highest base rating and source-key order. Use `source_cards` to preserve exact club-decade cards, for example `{'k': 'Q8682:1990', 'p': 'Q11571'}` if that record exists in the bundle.

`placement='best'` asks the actual engine to assign the eleven starters. `placement='ordered'` uses your input order: eleven formation slots followed by four bench places. The assignment helper maximises position/era-adjusted base ratings; it does not optimise chemistry or season wins. Manager and formation edits are notebook experiments.

In [ ]:
settings = {
    'seed': 20261006,
    'decade': 1990,
    'manager': None,  # A manager name from the catalogue, or None for a seeded choice.
    'formation': None,  # A formation name, or None to use that manager option.
    'person_ids': [],
    'source_cards': [],
    'placement': 'best',
}

def engine(settings):
    """Run the repository JavaScript bridge and return its JSON response."""
    p = subprocess.run([NODE, str(R / 'notebooks' / 'engine_bridge.mjs')],
                       input=json.dumps(settings), capture_output=True, text=True,
                       encoding='utf-8', cwd=R, timeout=120, check=False)
    if p.returncode:
        raise RuntimeError(p.stderr.strip() or f'Node exited with code {p.returncode}')
    return json.loads(p.stdout)

result = engine(settings)
print(result['mode'])
print('Manager:', result['manager']['nm'], 'Formation:', result['formation'])
print('Match parameters:')
print(json.dumps(result['parameters'], indent=2))
display(pd.DataFrame(result['squad']))
print('Source counts:', result['source_counts'])
print('Exact source cards for replay:')
print(json.dumps(result['source_cards']))


## See fit, era and rating changes

The starter table comes from the engine's `rate` result. Bench cards have no slot-fit loss. Source `f` is a matching edition, `n` is a nearby edition/club extrapolation, `i` is an EA reconstruction, and `e` is a fitted estimate. Timeless, duo/club links and manager effects are game rules; their existence does not establish a measured advantage.

In [ ]:
display(pd.DataFrame(result['starters']))
display(pd.DataFrame(result['bench']))
display(pd.DataFrame([result['lines']]))


## Play the league and European Cup

The league has twenty teams and a double round robin. The Cup has sixteen teams, two-legged ties until a neutral final, and extra time or penalties when needed. Historical club-decade pools and the same modern competition rules create a fictional season. Other clubs play each other through the same match engine.

In [ ]:
season = result['season']
display(pd.DataFrame(season['L']['tab']))
display(pd.DataFrame([
    {k: v for k, v in fixture.items() if k != 'M'} for fixture in season['L']['res']
]))
cup_rows = []
for round_ties in season['K']['rounds']:
    for t in round_ties:
        cup_rows.append({'round': t['k'], 'home': t['an'], 'away': t['bn'],
                         'aggregate': f"{t['agg'][0]}-{t['agg'][1]}", 'winner': t['wn'],
                         'extra_time': t.get('et', False), 'penalties': t.get('pw')})
display(pd.DataFrame(cup_rows))
print('Cup winner:', season['K']['champN'])
display(pd.DataFrame(result['player_stats']))
if 'awards' in result:
    print(json.dumps(result['awards'], indent=2, ensure_ascii=False))


## Replay and keep an inspectable result

This check reruns the same inputs and compares the entire bridge response. Replaying requires the same engine, data hash, settings, cards and parameters. A seed alone does not preserve a run after an update. Enable the export below only when you have chosen a persistent output folder; the default writes nothing.

In [ ]:
again = engine(settings)
assert result == again, 'The same settings produced a different bridge response.'
print('Deterministic replay matched for these settings and this data bundle.')

export_path = None  # Set a persistent Path to write the full result as JSON.
if export_path is not None:
    p = Path(export_path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Wrote:', p.resolve())


## Interpretation limits

Earlier decades depend on incomplete Wikidata records and estimated or reconstructed ratings. Whole-stint apps and goals are apportioned, coarse positions are mapped to modern roles, and decade squads can combine players who never shared a season. FIFA prediction errors evaluate the modern observed overlap; they do not validate historical ability. Read the loaded match-calibration report's actual holdout and baseline comparison before interpreting it.

Formation weights, cross-era loss, chemistry, tags, manager grades, absences and substitutions contain declared game assumptions. Modern calibration cannot establish these effects for mixed-era drafts. Goals, assists, clean sheets and awards in this notebook are simulated events. The model omits detailed pressing and defensive actions, and the same modern league/Cup rules are applied in every decade. The supplied pool covers male football and selected European clubs. These outputs describe this game, not a claim about who would win a real cross-era match.